In [ ]:
import torch, os
print("GPU available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
print("Files:", [f for f in os.listdir('/content') if f.endswith('.csv')])

GPU available: True
GPU: Tesla T4
Files: ['train.csv', 'liar_clean.csv', 'test.csv']


In [ ]:
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer

MODEL = "distilbert-base-uncased"
tok = AutoTokenizer.from_pretrained(MODEL)

train_df = pd.read_csv("/content/train.csv")[["text_bert", "label"]].dropna()
test_df  = pd.read_csv("/content/test.csv")[["text_bert", "label"]].dropna()

def tokenize(batch):
    return tok(batch["text_bert"], truncation=True, max_length=256)

train_ds = Dataset.from_pandas(train_df, preserve_index=False).map(tokenize, batched=True)
test_ds  = Dataset.from_pandas(test_df,  preserve_index=False).map(tokenize, batched=True)
print(train_ds); print(test_ds)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/50441 [00:00<?, ? examples/s]

Map:   0%|          | 0/12611 [00:00<?, ? examples/s]

Dataset({
    features: ['text_bert', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 50441
})
Dataset({
    features: ['text_bert', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 12611
})


In [ ]:
import numpy as np

from transformers import (AutoModelForSequenceClassification, TrainingArguments,
                          Trainer, DataCollatorWithPadding)
from sklearn.metrics import accuracy_score, f1_score

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL, num_labels=2, id2label={0: "REAL", 1: "FAKE"}, label2id={"REAL": 0, "FAKE": 1})

def metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    return {"accuracy": accuracy_score(p.label_ids, preds),
            "f1": f1_score(p.label_ids, preds)}

args = TrainingArguments(
    output_dir="/content/out",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    fp16=True,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=200,
    report_to="none",
)

trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=test_ds,
                  data_collator=DataCollatorWithPadding(tok), compute_metrics=metrics)
trainer.train()

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.039338,0.030235,0.990881,0.989795
2,0.013513,0.038202,0.992150,0.991210


TrainOutput(global_step=6306, training_loss=0.04485161269548542, metrics={'train_runtime': 738.0886, 'train_samples_per_second': 136.68, 'train_steps_per_second': 8.544, 'total_flos': 6681788055607296.0, 'train_loss': 0.04485161269548542, 'epoch': 2.0})

In [ ]:
# 1. Save the model + tokenizer
trainer.save_model("/content/distilbert")
tok.save_pretrained("/content/distilbert")

# 2. Cross-dataset test on LIAR
liar_df = pd.read_csv("/content/liar_clean.csv")[["text_bert", "label"]].dropna()
liar_ds = Dataset.from_pandas(liar_df, preserve_index=False).map(tokenize, batched=True)
out = trainer.predict(liar_ds)
print("LIAR results:", {k: round(v, 4) for k, v in out.metrics.items() if "accuracy" in k or "f1" in k})

# 3. Zip and download
!cd /content && zip -qr distilbert.zip distilbert
from google.colab import files
files.download("/content/distilbert.zip")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Map:   0%|          | 0/12810 [00:00<?, ? examples/s]

LIAR results: {'test_accuracy': 0.5191, 'test_f1': 0.4492}


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files
files.download("/content/distilbert.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>